# Deploy NWDAF Dashboard to Google Cloud
This notebook automates deploying your React UI directly onto your GCP VM.
It installs **Nginx**, downloads the `.jsx` file from GitHub, automatically transforms it to run natively in the browser without a build step, and sets up an API proxy to bypass CORS issues.

In [ ]:
# 1. Authenticate to Google Cloud
from google.colab import auth
auth.authenticate_user()
print('✅ Authenticated successfully!')

In [ ]:
# 2. Configure GCP Settings & GitHub URL
PROJECT_ID = 'g-ai-lab-491619'
ZONE = 'europe-west4-a'
VM_NAME = 'open5gs-ai-lab'

# ⚠️ UPDATE THIS TO YOUR EXACT RAW GITHUB URL if different
GITHUB_JSX_URL = 'https://raw.githubusercontent.com/cem8kaya/open5gs-nwdaf/main/nwdaf_dashboard.jsx'

!gcloud config set project {PROJECT_ID}
!gcloud config set compute/zone {ZONE}

In [ ]:
%%writefile deploy_ui.sh
#!/bin/bash

echo "Installing Nginx..."
sudo apt-get update -qq
sudo apt-get install -y nginx -qq

echo "Downloading React file from GitHub..."
sudo curl -sL "$1" -o /var/www/html/nwdaf_dashboard.jsx

echo "Generating index.html wrapper..."
sudo tee /var/www/html/index.html > /dev/null << 'HTML_EOF'
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>NWDAF Operations Dashboard</title>
  <script src="https://cdn.tailwindcss.com"></script>
  <script type="importmap">
  {
    "imports": {
      "react": "https://esm.sh/react@18.2.0",
      "react-dom/client": "https://esm.sh/react-dom@18.2.0/client",
      "recharts": "https://esm.sh/recharts@2.12.3?deps=react@18.2.0,react-dom@18.2.0",
      "lucide-react": "https://esm.sh/lucide-react@0.359.0?deps=react@18.2.0"
    }
  }
  </script>
  <script src="https://unpkg.com/@babel/standalone/babel.min.js"></script>
</head>
<body class="bg-[#0a0e1a] text-white">
  <div id="root">
    <div style="display:flex; height:100vh; align-items:center; justify-content:center; flex-direction:column; gap:1rem;">
      <div class="animate-spin rounded-full h-12 w-12 border-b-2 border-blue-500"></div>
      <p class="text-gray-400 font-mono">Compiling React Dashboard...</p>
    </div>
  </div>
  <script type="module">
    import React from 'react';
    import { createRoot } from 'react-dom/client';
    import * as Recharts from 'recharts';
    import * as LucideReact from 'lucide-react';
    window.React = React;
    window.recharts = Recharts;
    window.lucideReact = LucideReact;
    window.NWDAF_BASE_URL = window.location.origin + "/nwdaf-analytics/v1";
    async function loadApp() {
      try {
        const res = await fetch('./nwdaf_dashboard.jsx');
        if (!res.ok) throw new Error("Failed to fetch JSX file");
        let jsxCode = await res.text();
        jsxCode = jsxCode.replace(/import\s+React,\s*\{([^}]+)\}\s+from\s+['"]react['"];?/g, (match, p1) => { return `const { ${p1} } = window.React;`; });
        jsxCode = jsxCode.replace(/import\s+React\s+from\s+['"]react['"];?/g, '');
        jsxCode = jsxCode.replace(/import\s+\{([^}]+)\}\s+from\s+['"]recharts['"];?/g, (match, p1) => {
          return `const { ${p1.replace(/\bas\b/g, ':').replace(/\s+/g, ' ')} } = window.recharts;`;
        });
        jsxCode = jsxCode.replace(/import\s+\{([^}]+)\}\s+from\s+['"]lucide-react['"];?/g, (match, p1) => {
          return `const { ${p1.replace(/\bas\b/g, ':').replace(/\s+/g, ' ')} } = window.lucideReact;`;
        });
        jsxCode = jsxCode.replace(/export\s+default\s+function\s+App/, 'window.App = function App');
        const compiled = Babel.transform(jsxCode, { presets: ['react'] }).code;
        eval(compiled);
        const root = createRoot(document.getElementById('root'));
        root.render(React.createElement(window.App));
      } catch (err) {
        document.getElementById('root').innerHTML = `<div style="padding:2rem;color:red;font-family:monospace;"><h3>Failed to load App</h3><p>${err.message}</p></div>`;
        console.error(err);
      }
    }
    loadApp();
  </script>
</body>
</html>
HTML_EOF

echo "Configuring Nginx Reverse Proxy for API..."
sudo tee /etc/nginx/sites-available/default > /dev/null << 'NGINX_EOF'
server {
    listen 80 default_server;
    root /var/www/html;
    index index.html;
    server_name _;
    location / {
        try_files $uri $uri/ =404;
    }
    location /nwdaf-analytics/ {
        proxy_pass http://127.0.0.1:7779/nwdaf-analytics/;
    }
}
NGINX_EOF

sudo systemctl restart nginx
echo "✅ Nginx successfully configured and restarted!"


In [ ]:
# 4. Run the deployment script on the VM
!echo "Deploying to VM..."
!gcloud compute ssh {VM_NAME} --command="bash -s {GITHUB_JSX_URL}" < deploy_ui.sh

In [ ]:
# 5. Open HTTP Firewall Port (Port 80)
FIREWALL_RULE_NAME = 'allow-http-80'
rule_exists = !gcloud compute firewall-rules list --filter="name={FIREWALL_RULE_NAME}" --format="value(name)"

if not rule_exists:
    !gcloud compute firewall-rules create {FIREWALL_RULE_NAME} \
        --direction=INGRESS \
        --priority=1000 \
        --network=default \
        --action=ALLOW \
        --rules=tcp:80 \
        --source-ranges=0.0.0.0/0 \
        --target-tags=http-server
    !gcloud compute instances add-tags {VM_NAME} --tags=http-server
    print('✅ Firewall rule for HTTP (80) created.')
else:
    print('✅ HTTP Firewall rule already exists.')

In [ ]:
# 6. Get your Dashboard URL!
external_ip = !gcloud compute instances describe {VM_NAME} --format="get(networkInterfaces[0].accessConfigs[0].natIP)"
if external_ip:
    ip = external_ip[0]
    print("="*60)
    print("🚀 DEPLOYMENT COMPLETE")
    print("="*60)
    print(f"\n🌐 Your Dashboard is now live at: http://{ip}\n")
    print("Note: Since the API proxy is on the same server, you will NOT have CORS issues.")
    print("="*60)
